## Traffic Demand Prediction Model
Loading the dependencies for the task.

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

### Load the Datasets
Reading `train.csv` and `test.csv`.

In [2]:
train = pd.read_csv("./dataset/train.csv")
test = pd.read_csv("./dataset/test.csv")

print("Train size:", train.shape)
print("Test size:", test.shape)

Train size: (77299, 11)
Test size: (41778, 10)


### Feature Engineering
Parsing time fields, deriving `time_in_mins`, `day_of_week` and setting default categorical/numerical imputations to prevent target leaks.

In [3]:
# Derive dictionaries for lag features from day 48
train_d48 = train[train['day'] == 48]
dict_exact_time = train_d48.groupby(['geohash', 'timestamp'])['demand'].mean().to_dict()
dict_geohash = train_d48.groupby('geohash')['demand'].mean().to_dict()

def preprocess(df):
    df_new = df.copy()
    
    # Time features
    df_new['hour'] = df_new['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df_new['minute'] = df_new['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df_new['time_in_mins'] = df_new['hour'] * 60 + df_new['minute']
    
    # Geohash aggregation features
    df_new['geohash_5'] = df_new['geohash'].str[:5]
    df_new['geohash_4'] = df_new['geohash'].str[:4]
    
    # Lag features
    keys = list(zip(df_new['geohash'], df_new['timestamp']))
    df_new['lag_1d_exact'] = [dict_exact_time.get(k, np.nan) for k in keys]
    df_new['lag_1d_geohash_mean'] = df_new['geohash'].map(dict_geohash)
    
    # Geohash-level imputations for continuous variables
    df_new['Temperature'] = df_new['Temperature'].fillna(df_new.groupby('geohash')['Temperature'].transform('mean'))
    df_new['Temperature'] = df_new['Temperature'].fillna(df_new['Temperature'].mean()) # global fallback
    
    df_new['NumberofLanes'] = df_new['NumberofLanes'].fillna(df_new.groupby('geohash')['NumberofLanes'].transform('mean'))
    df_new['NumberofLanes'] = df_new['NumberofLanes'].fillna(df_new['NumberofLanes'].mean())
    df_new['NumberofLanes'] = df_new['NumberofLanes'].fillna(1) # ultimate fallback
    
    # Geohash-level imputations for categorical variables
    for col in ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']:
        if col in df_new.columns:
            mode_series = df_new.groupby('geohash')[col].transform(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
            df_new[col] = df_new[col].fillna(mode_series)
            df_new[col] = df_new[col].fillna('Unknown')
            
    df_new['day_of_week'] = df_new['day'] % 7
    df_new['is_weekend'] = (df_new['day_of_week'] >= 5).astype(int)
    
    return df_new

all_train = preprocess(train)
test_df = preprocess(test)

y_train = all_train['demand']
X_train = all_train.drop(['demand', 'Index', 'timestamp', 'day'], axis=1)

X_test_ids = test_df['Index']
X_test = test_df.drop(['Index', 'timestamp', 'day'], axis=1)

### K-Fold Cross Validation Setup
Training CatBoost using 5 independent folds to maximize evaluation `R2` metrics.

In [4]:
categorical_features = ['geohash', 'geohash_5', 'geohash_4', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X_train))

print("Starting 5-Fold Training with Lags...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"--- Fold {fold+1} ---")
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.08,
        depth=10, 
        l2_leaf_reg=5,
        loss_function='RMSE',
        eval_metric='R2',
        random_seed=42 + fold,
        early_stopping_rounds=150,
        verbose=1000
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        cat_features=categorical_features,
        use_best_model=True
    )
    
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / kf.n_splits

oof_r2 = r2_score(y_train, oof_preds)
print(f"\nOverall Out-Of-Fold R2 Score: {oof_r2:.6f}")

Starting 5-Fold Training with Lags...
--- Fold 1 ---
0:	learn: 0.1439387	test: 0.1430855	best: 0.1430855 (0)	total: 148ms	remaining: 7m 25s
1000:	learn: 0.9982556	test: 0.9916046	best: 0.9916079 (994)	total: 1m 3s	remaining: 2m 7s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9916158311
bestIteration = 1099

Shrink model to first 1100 iterations.
--- Fold 2 ---
0:	learn: 0.1456508	test: 0.1455208	best: 0.1455208 (0)	total: 66ms	remaining: 3m 17s
1000:	learn: 0.9982343	test: 0.9929616	best: 0.9929618 (979)	total: 1m 1s	remaining: 2m 2s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9930212054
bestIteration = 1296

Shrink model to first 1297 iterations.
--- Fold 3 ---
0:	learn: 0.1407932	test: 0.1401347	best: 0.1401347 (0)	total: 15ms	remaining: 45s
1000:	learn: 0.9983348	test: 0.9926878	best: 0.9927083 (911)	total: 1m	remaining: 2m
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9927082501
bestIteration = 911

Shrink mode

### Generate Submission File
Clipping variables to limits `(0, None)` as demand can't be negative, and saving.

In [5]:
submission = pd.DataFrame({
    'Index': X_test_ids,
    'demand': np.clip(test_preds, 0, None)
})

submission.to_csv("submission.csv", index=False)
print("Submission saved!")

Submission saved!


In [6]:
# Calculate actual evaluation metric score
final_score = max(0, 100 * r2_score(y_train, oof_preds))
print(f"Final Evaluation Score: {final_score:.4f}")

Final Evaluation Score: 99.2366
